In [43]:
from rdkit.Chem import Descriptors, MolFromSmiles
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score, matthews_corrcoef, f1_score
from pandas import DataFrame, concat
from pickle import load

In [12]:
#создаем словарь из дескприторов структуры
ConstDescriptors = {"HeavyAtomCount" : Descriptors.HeavyAtomCount,
                       "NHOHCount" : Descriptors.NHOHCount,
                       "NOCount" : Descriptors.NOCount,
                       "NumHAcceptors" : Descriptors.NumHAcceptors,
                       "NumHDonors" : Descriptors.NumHDonors,
                       "NumHeteroatoms" : Descriptors.NumHeteroatoms,
                       "NumRotatableBonds" : Descriptors.NumRotatableBonds,
                       "NumValenceElectrons" : Descriptors.NumValenceElectrons,
                       "NumAromaticRings" : Descriptors.NumAromaticRings,
                       "NumAliphaticHeterocycles" : Descriptors.NumAliphaticHeterocycles,
                       "RingCount" : Descriptors.RingCount}

#создаем словарь из физико-химических дескрипторов                            
PhisChemDescriptors = {"MW" : Descriptors.MolWt,
                          "LogP" : Descriptors.MolLogP,
                          "MR" : Descriptors.MolMR,
                          "TPSA" : Descriptors.TPSA}

#объединяем все дескрипторы в один словарь
descriptors = {}
descriptors.update(ConstDescriptors)
descriptors.update(PhisChemDescriptors)
print(len(descriptors), "количество дескрипторов в словаре")


#функция для генерации дескрипторов из молекул
def mol_dsc_calc(mols):
    
    return DataFrame({k: f(m) for k, f in descriptors.items()} 
             for m in mols)


with open('../data/classifier/modeling_data.pickle', 'rb') as file:
    data = load(file)
#оформляем sklearn трансформер для использования в конвеерном моделировании (sklearn Pipeline)
descriptors_transformer = FunctionTransformer(mol_dsc_calc, validate=False)

15 количество дескрипторов в словаре


In [13]:
molecules = [
    MolFromSmiles(mol) for mol in data['SMILES']
]

In [14]:
X = descriptors_transformer.transform(molecules)
Y = data['Activity']
XY = concat((X, Y), axis=1)

In [15]:
XY

,HeavyAtomCount,NHOHCount,NOCount,NumHAcceptors,NumHDonors,NumHeteroatoms,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumAliphaticHeterocycles,RingCount,MW,LogP,MR,TPSA,Activity
0,22,0,1,1,0,4,3,116,2,0,3,305.343,4.82470,80.4150,3.24,1
1,33,2,8,4,2,8,8,174,2,2,4,450.539,2.19090,125.1194,90.98,1
2,22,1,6,6,1,6,3,116,2,2,4,298.394,1.15590,83.2177,58.87,1
3,24,1,4,4,1,4,3,128,2,1,4,325.456,4.01342,96.3128,49.25,1
4,23,1,5,5,1,6,3,122,2,2,4,315.396,1.90000,85.3807,45.98,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75924,17,3,3,2,3,3,0,88,2,1,4,228.295,1.53040,67.1582,48.05,1
75925,24,1,4,3,1,5,3,124,2,2,4,327.355,2.32120,85.9918,49.77,1
75926,23,1,5,4,1,6,2,126,1,2,3,323.364,2.73460,81.6418,59.00,0
75927,23,0,5,4,0,6,3,122,1,2,4,319.332,2.05930,78.1170,55.84,1


In [16]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.15)

In [17]:
scaler_x = StandardScaler().fit(x_train)
x_train_scal = scaler_x.transform(x_train)
x_test_scal = scaler_x.transform(x_test)

In [38]:
x_train_scal

array([[ 2.17810431, -0.07344431,  1.04767084, ...,  0.56789973,
         1.86309566,  0.68393714],
       [-2.05921022,  0.96603425, -0.83540568, ..., -2.22951898,
        -1.84045078, -0.86401607],
       [-0.17595932,  2.00551282,  0.41997867, ..., -0.43999552,
         0.14219879,  0.83524084],
       ...,
       [ 0.05944705, -0.07344431, -0.2077135 , ...,  0.46420387,
         0.03822672, -0.22522798],
       [ 0.05944705, -0.07344431,  0.41997867, ..., -0.17343479,
         0.02955678,  0.49145019],
       [ 1.23647886, -0.07344431,  1.04767084, ...,  0.32275736,
         1.02584236,  1.22603412]], shape=(64539, 15))

In [ ]:
rf = RandomForestClassifier()
pg = {'n_estimators': [50, 100, 200, 300, 400, 500],
      'max_depth': [None, 1, 3, 5, 7, 10],
      }
cv = RepeatedKFold(n_splits=5, n_repeats=5)
gs = GridSearchCV(rf, pg, verbose=3, cv=cv)

gs.fit(x_train_scal, y_train)
gs.best_params_

Fitting 50 folds for each of 60 candidates, totalling 3000 fits
[CV 1/50] END .....max_depth=1, n_estimators=50;, score=0.899 total time=   0.9s
[CV 2/50] END .....max_depth=1, n_estimators=50;, score=0.896 total time=   0.7s
[CV 3/50] END .....max_depth=1, n_estimators=50;, score=0.893 total time=   0.7s
[CV 4/50] END .....max_depth=1, n_estimators=50;, score=0.895 total time=   0.7s
[CV 5/50] END .....max_depth=1, n_estimators=50;, score=0.897 total time=   0.7s
[CV 6/50] END .....max_depth=1, n_estimators=50;, score=0.895 total time=   0.7s
[CV 7/50] END .....max_depth=1, n_estimators=50;, score=0.899 total time=   0.7s
[CV 8/50] END .....max_depth=1, n_estimators=50;, score=0.897 total time=   0.7s
[CV 9/50] END .....max_depth=1, n_estimators=50;, score=0.899 total time=   0.7s
[CV 10/50] END ....max_depth=1, n_estimators=50;, score=0.891 total time=   1.0s
[CV 11/50] END ....max_depth=1, n_estimators=50;, score=0.898 total time=   1.5s
[CV 12/50] END ....max_depth=1, n_estimators=

In [39]:
y_pred = gs.predict(x_test_scal)

In [31]:
gs.best_estimator_

RandomForestClassifier(max_depth=10, n_estimators=500)

In [50]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel().tolist()
precision = tp / (tp + fp)
recall = tp / (tp + fn)

In [54]:
print(f'ACC = {accuracy_score(y_test, y_pred):.3f}')
print(f'BA = {balanced_accuracy_score(y_test, y_pred):.3f}')
print(f'MCC = {matthews_corrcoef(y_test, y_pred):.3f}')
print(f'ROC AUC = {roc_auc_score(y_test, y_pred):.3f}')
print(f'F1 = {f1_score(y_test, y_pred):.3f}')
print(f'Precision = {precision:.3f}')
print(f'Recall = {recall:.3f}')

ACC = 0.922
BA = 0.657
MCC = 0.471
ROC AUC = 0.657
F1 = 0.958
Precision = 0.928
Recall = 0.990
